# Build DPO Dataset from FunSearch Runs

This notebook constructs a DPO preference dataset for the **Capacitated Vehicle Routing Problem (CVRP)**
from the algorithm search results produced by `start_funsearch.sh`.

Pipeline:
1. Load raw `(program, score)` pairs from `results/cvrp/funsearch/run{1,2,3}/algo/`
2. Merge and deduplicate across runs
3. Apply DAR sampling (`DatasetCreator`) to build preference pairs `(prompt, chosen, rejected)`
4. Save the dataset as a `.pkl` file for DPO training

In [ ]:
import copy
import glob
import os
import pickle

from dataset_creator import DatasetCreator

## 1. Prompt and Code Template

The prompt describes the task and the expected function signature.
It is used as the `x` (input) side of every preference pair `(x, y+, y-)`.

In [ ]:
prompt = """\
Your task is to design a `select_next_node` function to solve capacitated vehicle routing problem (CVRP). \
The task is to design a novel algorithm to select the next node in each step, \
with the objective of minimizing the total cost.

```python
import numpy as np

def select_next_node(current_node: int, depot: int, unvisited_nodes: np.ndarray, rest_capacity: np.ndarray, demands: np.ndarray, distance_matrix: np.ndarray) -> int:
    \"\"\"Design a novel algorithm to select the next node in each step.
    Args:
        current_node: ID of the current node.
        depot: ID of the depot.
        unvisited_nodes: Array of IDs of unvisited nodes.
        rest_capacity: rest capacity of vehicle
        demands: demands of nodes
        distance_matrix: Distance matrix of nodes.
    Return:
        ID of the next node to visit.
    \"\"\"
    return unvisited_nodes[0]
```

Please design a novel `select_next_node` function that follows the template below. Please only output the function implementation.

import numpy as np

def select_next_node(current_node: int, depot: int, unvisited_nodes: np.ndarray, rest_capacity: np.ndarray, demands: np.ndarray, distance_matrix: np.ndarray) -> int:
    \"\"\"Design a novel algorithm to select the next node in each step.
    Args:
        current_node: ID of the current node.
        depot: ID of the depot.
        unvisited_nodes: Array of IDs of unvisited nodes.
        rest_capacity: rest capacity of vehicle
        demands: demands of nodes
        distance_matrix: Distance matrix of nodes.
    Return:
        ID of the next node to visit.
    \"\"\"
    # your implementations here...
"""

template = """\
import numpy as np

def select_next_node(current_node: int, depot: int, unvisited_nodes: np.ndarray, rest_capacity: np.ndarray, demands: np.ndarray, distance_matrix: np.ndarray) -> int:
    \"\"\"Design a novel algorithm to select the next node in each step.
    Args:
        current_node: ID of the current node.
        depot: ID of the depot.
        unvisited_nodes: Array of IDs of unvisited nodes.
        rest_capacity: rest capacity of vehicle
        demands: demands of nodes
        distance_matrix: Distance matrix of nodes.
    Return:
        ID of the next node to visit.
    \"\"\"
    return unvisited_nodes[0]
"""

## 2. Load Algorithm Search Results

FunSearch logs each sampled algorithm as a dict with keys `program` (code string)
and `score` (evaluator result) into batched `.pkl` files under `{logdir}/algo/`.
We read all batches from each run and convert to the `{func, score}` format
expected by `DatasetCreator`.

In [ ]:
def load_data_from_run(run_dir: str) -> list[dict]:
    """Load all (program, score) pairs from a single FunSearch run directory."""
    algo_dir = os.path.join(run_dir, "algo")
    pkl_files = sorted(
        glob.glob(os.path.join(algo_dir, "*.pkl")),
        key=lambda p: int(os.path.splitext(os.path.basename(p))[0]),
    )
    data = []
    for path in pkl_files:
        with open(path, "rb") as f:
            batch = pickle.load(f)  # list of dicts
        for item in batch:
            score = item.get("score")
            program = item.get("program")
            if score is not None and program:
                data.append({"func": program, "score": score})
    return data


run_dirs = [
    "results/cvrp/funsearch/run1",
    "results/cvrp/funsearch/run2",
    "results/cvrp/funsearch/run3",
]

all_data = []
for run_dir in run_dirs:
    run_data = load_data_from_run(run_dir)
    print(f"{run_dir}: {len(run_data)} samples")
    all_data.extend(run_data)

print(f"\nTotal samples across all runs: {len(all_data)}")

## 3. Build Preference Dataset (DAR Sampling)

`DatasetCreator` implements the Diversity-Aware Rank-based (DAR) sampling strategy
(paper Section 2.2): it partitions the algorithm pool into `M` fitness subsets and
samples `(chosen, rejected)` pairs with a temperature-controlled bias toward
higher-quality algorithms, while enforcing a minimum quality gap between the two.

In [ ]:
creator = DatasetCreator(
    data=copy.deepcopy(all_data),
    prompt=prompt,
    number_of_subsets=10,  # M: number of fitness partitions
    p=3,                   # τ: sampling temperature (smaller = stronger bias to top subsets)
    remove_duplicate=True,
)
print(f"\nUnique algorithms after deduplication: {len(creator)}")

In [ ]:
dataset = creator.create_dataset(
    npairs=1000,        # number of unique preference pairs to sample
    dataset_size=10000, # duplicate up to this size for training
)
print(f"Dataset size: {len(dataset)}")
print("\nExample pair:")
print(dataset[0])

## 4. Save Dataset

In [ ]:
os.makedirs("results/datasets", exist_ok=True)
save_path = "results/datasets/cvrp_funsearch_dpo.pkl"

with open(save_path, "wb") as f:
    pickle.dump(dataset, f)

print(f"Saved {len(dataset)} pairs to {save_path}")